# 🔁 Model 3: Bidirectional LSTM

**Input:** `(batch, N_BANDS, n_channels)` — bands as sequence  
**Feature output:** 128-D penultimate vector → SVM

## 0. Mount Google Drive
Required since this notebook reads/writes files under `/content/drive/MyDrive/...`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import subprocess
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or '⚠️ No GPU')

Wed Sep 30 19:12:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install scikit-learn openpyxl -q

In [4]:
import os,gc,time,numpy as np,pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers,models,Input,optimizers as opt_module

BAND_NAMES=["Delta","Theta","Alpha","Beta","Gamma"]

def load_dataset(p):
    # Read the file depending on its format (Excel or CSV)
    df = pd.read_excel(p) if p.endswith(".xlsx") else pd.read_csv(p)
    # Identify columns containing float values and convert them to float32
    # to reduce memory usage
    float_cols = df.select_dtypes(include=["float64"]).columns
    df[float_cols] = df[float_cols].astype(np.float32)
    # NaN guard: a handful of windows can contain NaN feature values.
    n_nan = df.isna().any(axis=1).sum()
    if n_nan:
        print(f"    [nan guard] dropping {n_nan} rows with NaN features")
        df = df.dropna().reset_index(drop=True)
    return df

def normalize_tensors(Xtr, Xv, Xte):
    """Fit mean/std on TRAIN split only (per-feature), apply to all splits."""
    mean = Xtr.mean(axis=0, keepdims=True)
    std  = Xtr.std(axis=0, keepdims=True)
    Xtr  = (Xtr - mean) / (std + 1e-8)
    Xv   = (Xv  - mean) / (std + 1e-8)
    Xte  = (Xte - mean) / (std + 1e-8)
    return Xtr, Xv, Xte
def get_band_columns(df): return {b:[c for c in df.columns if c.startswith(f"{b}_")] for b in BAND_NAMES}
def prepare_tensors_seq(df):
    bc=get_band_columns(df)
    X=np.stack([df[bc[b]].values for b in BAND_NAMES],axis=1)  # (N,5,n_ch)
    return X, df["label"].values.astype(int)
def evaluate_with_svm(Xtr, ytr, Xte, yte, max_train=20000):
    # RBF-kernel SVC scales ~O(n^2)-O(n^3) in sample count -- subsample.
    if len(Xtr) > max_train:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(Xtr), max_train, replace=False)
        Xtr, ytr = Xtr[idx], ytr[idx]
    sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
    svm = SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", random_state=42)
    svm.fit(Xtr, ytr); p = svm.predict(Xte)

    tn, fp, fn, tp = confusion_matrix(yte, p, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    return {
        "accuracy": round(accuracy_score(yte, p), 4),
        "sensitivity": round(sensitivity, 4),
        "specificity": round(specificity, 4),
        "f1_macro": round(f1_score(yte, p, average="macro"), 4),
        "f1_weighted": round(f1_score(yte, p, average="weighted"), 4),
        "report": classification_report(yte, p, labels=[0, 1], target_names=["No Seizure", "Seizure"])
    }
def save_results(row,path):
    os.makedirs(os.path.dirname(path),exist_ok=True)
    pd.DataFrame([row]).to_csv(path,mode="a",header=not os.path.exists(path),index=False)
    print("Saved:",row)
def compile_model(m, opt):
    m.compile(optimizer=opt, loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
                       tf.keras.metrics.Precision(name="precision"),
                       tf.keras.metrics.Recall(name="recall")])
    return m


In [5]:
import tensorflow as tf
import numpy as np

# 1. Ensure reproducibility (Crucial for LSTMs)
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# 2. Prevent GPU Out-Of-Memory (OOM) errors
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

In [6]:
DATASET_A  = "/content/drive/MyDrive/epilepsy_outputs/dataset_A.csv"
DATASET_B  = "/content/drive/MyDrive/epilepsy_outputs/dataset_B.csv"
RESULTS    = "/content/drive/MyDrive/epilepsy_outputs/results.csv"
OUTPUT_DIR = "/content/drive/MyDrive/epilepsy_outputs"
EPOCHS=50; BATCH_SIZE=64; PATIENCE=8; SEED=42
MODEL_NAME = "lstm"

## Architecture

```
Input -> BiLSTM(128, return_seq) -> BN -> Dropout
      -> BiLSTM(64,  return_seq) -> BN -> Dropout
      -> BiLSTM(64)              -> BN
      -> Dense(128, ReLU) ← feature vector -> Dense(1, sigmoid)
```

In [7]:
from tensorflow.keras import regularizers

# Updated LSTM Architecture
def build_lstm(n_bands=5, n_channels=18, units=[128, 64, 64], feature_dim=128, dropout=0.3, l2=1e-4, output_bias=None):
    bias_init = tf.keras.initializers.Constant(output_bias) if output_bias is not None else "zeros"
    inp = layers.Input(shape=(n_bands, n_channels), name="eeg_input")
    x = inp

    for i, u in enumerate(units):
        ret = (i < len(units) - 1)
        # FIX #13: recurrent_dropout forces TF off the fast cuDNN LSTM kernel
        # onto a much slower unrolled implementation (~15min/epoch vs ~1-2min).
        # With only 5 timesteps (the 5 frequency bands) here, recurrent_dropout's
        # regularization benefit -- designed for long sequences -- is minimal;
        # removed for computational tractability. Regular `dropout` (input-level)
        # is kept, so regularization is reduced, not eliminated.
        lstm = layers.LSTM(u, return_sequences=ret, dropout=dropout,
                           kernel_regularizer=regularizers.l2(l2),
                           recurrent_regularizer=regularizers.l2(l2), name=f"lstm_{i}")

        x = layers.Bidirectional(lstm, name=f"bilstm_{i}")(x)
        x = layers.BatchNormalization(name=f"bn_{i}")(x)
        if ret:
            x = layers.Dropout(dropout, name=f"drop_{i}")(x)

    feat = layers.Dense(feature_dim, activation="relu", name="feature_vector",
                        kernel_regularizer=regularizers.l2(l2))(x)
    feat = layers.Dropout(dropout)(feat)
    out = layers.Dense(1, activation="sigmoid", name="output", bias_initializer=bias_init)(feat)

    return models.Model(inp, out, name="LSTM"), models.Model(inp, feat, name="LSTM_Feat")

In [8]:
model, feat_ext = build_lstm()
model.summary()
print("Feature extractor output:", feat_ext.output_shape)

# Clean up the dummy graph before starting the real training loop
del model, feat_ext
tf.keras.backend.clear_session()

Model: "LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ eeg_input (InputLayer)          │ (None, 5, 18)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm_0 (Bidirectional)        │ (None, 5, 256)         │       150,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_0 (BatchNormalization)       │ (None, 5, 256)         │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_0 (Dropout)                │ (None, 5, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm_1 (Bidirectional)        │ (None, 5, 128)         │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_1 (BatchNormalization)       │ (None, 5, 128)         │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_1 (Dropout)                │ (None, 5, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm_2 (Bidirectional)        │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_2 (BatchNormalization)       │ (None, 128)            │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_vector (Dense)          │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 432,385 (1.65 MB)

 Trainable params: 431,361 (1.65 MB)

 Non-trainable params: 1,024 (4.00 KB)

Feature extractor output: (None, 128)


In [9]:
def run_trial(ds_path, ds_label, opt_name, opt_obj):
    print(f"\n{'='*55}\n  {MODEL_NAME} | Dataset {ds_label} | {opt_name}\n{'='*55}")
    df=load_dataset(ds_path); X,y=prepare_tensors_seq(df)
    n_bands,n_ch=X.shape[1],X.shape[2]
    Xtrv,Xte,ytrv,yte=train_test_split(X,y,test_size=0.15,stratify=y,random_state=SEED)
    Xtr,Xv,ytr,yv=train_test_split(Xtrv,ytrv,test_size=0.15/0.85,stratify=ytrv,random_state=SEED)
    Xtr,Xv,Xte=normalize_tensors(Xtr,Xv,Xte)   # train-only normalization
    total=len(ytr); n1=ytr.sum(); cw={0:total/(2*(total-n1)),1:total/(2*n1)}
    output_bias=np.log(n1/(total-n1))          # prior-informed bias init
    full_m,feat_m=build_lstm(n_bands=n_bands,n_channels=n_ch,output_bias=output_bias)
    full_m=compile_model(full_m,opt_obj)
    ckpt=os.path.join(OUTPUT_DIR,"checkpoints",f"{MODEL_NAME}_{ds_label}_{opt_name}_best.keras")
    os.makedirs(os.path.dirname(ckpt),exist_ok=True)
    cbs=[tf.keras.callbacks.EarlyStopping(monitor="val_auc",mode="max",patience=PATIENCE,restore_best_weights=True,verbose=1),
         tf.keras.callbacks.ModelCheckpoint(ckpt,monitor="val_auc",mode="max",save_best_only=True,verbose=0),
         tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=4,min_lr=1e-6,verbose=1)]
    t0=time.time()
    hist=full_m.fit(Xtr,ytr,validation_data=(Xv,yv),epochs=EPOCHS,batch_size=BATCH_SIZE,
                    class_weight=cw,callbacks=cbs,verbose=1)
    elapsed=round((time.time()-t0)/60,2)
    best_auc=max(hist.history.get("val_auc",[0]))
    print(f"  Done {elapsed}min | Best AUC:{best_auc:.4f}")
    f_tr=feat_m.predict(Xtr,batch_size=BATCH_SIZE,verbose=0)
    f_te=feat_m.predict(Xte,batch_size=BATCH_SIZE,verbose=0)
    res=evaluate_with_svm(f_tr,ytr,f_te,yte)
    print(f"  SVM Acc:{res['accuracy']} F1:{res['f1_macro']}\n",res["report"])
    save_results({"model":MODEL_NAME,"dataset":f"Dataset_{ds_label}","optimizer":opt_name,
                  "best_val_auc":round(best_auc,4),**{k:v for k,v in res.items() if k!="report"},
                  "train_time_min":elapsed},RESULTS)
    del full_m,feat_m; gc.collect(); tf.keras.backend.clear_session()

# RESUME SUPPORT: skip any (dataset, optimizer) combo already saved in
# RESULTS, so a runtime disconnect doesn't force a full restart from scratch.
already_done = set()
if os.path.exists(RESULTS):
    _prev = pd.read_csv(RESULTS)
    _prev_this_model = _prev[_prev["model"] == MODEL_NAME]
    already_done = set(zip(_prev_this_model["dataset"], _prev_this_model["optimizer"]))
    if already_done:
        print(f"[resume] found {len(already_done)} completed trial(s) already in {RESULTS}, skipping those:")
        for d, o in sorted(already_done):
            print(f"    - {d} | {o}")

for ds_label,ds_path in [("A",DATASET_A),("B",DATASET_B)]:
    for opt_name, opt_obj in {
        "Adam": opt_module.Adam(1e-3, clipnorm=1.0),
        "Nadam": opt_module.Nadam(1e-3, clipnorm=1.0),
        "AdamW": opt_module.AdamW(1e-3, weight_decay=1e-4, clipnorm=1.0)
    }.items():
        if (f"Dataset_{ds_label}", opt_name) in already_done:
            print(f"[resume] skipping {MODEL_NAME} | Dataset {ds_label} | {opt_name} (already done)")
            continue
        run_trial(ds_path,ds_label,opt_name,opt_obj)
print(f"\n✅ {MODEL_NAME} all trials done.")


[resume] found 5 completed trial(s) already in /content/drive/MyDrive/epilepsy_outputs/results.csv, skipping those:
    - Dataset_A | Adam
    - Dataset_A | AdamW
    - Dataset_A | Nadam
    - Dataset_B | Adam
    - Dataset_B | Nadam
[resume] skipping lstm | Dataset A | Adam (already done)
[resume] skipping lstm | Dataset A | Nadam (already done)
[resume] skipping lstm | Dataset A | AdamW (already done)
[resume] skipping lstm | Dataset B | Adam (already done)
[resume] skipping lstm | Dataset B | Nadam (already done)

  lstm | Dataset B | AdamW
Epoch 1/50
1798/1798 ━━━━━━━━━━━━━━━━━━━━ 59s 25ms/step - accuracy: 0.9987 - auc: 0.5872 - loss: 2.7463 - precision: 0.0444 - recall: 0.0556 - val_accuracy: 0.9994 - val_auc: 0.4064 - val_loss: 0.0744 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 2/50
1798/1798 ━━━━━━━━━━━━━━━━━━━━ 45s 25ms/step - accuracy: 0.9985 - auc: 0.6502 - loss: 2.2052 - precision: 0.0541 - recall: 0.0833 - val_accuracy: 0.9994 - val_au